In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report , f1_score
from collections import Counter

In [4]:
df = pd.read_csv('creditcard.csv')
print('class distribution')
print(df['Class'].value_counts())
print(f"\nPercentages:\n{df['Class'].value_counts(normalize=True)*100}")

class distribution
Class
0    284315
1       492
Name: count, dtype: int64

Percentages:
Class
0    99.827251
1     0.172749
Name: proportion, dtype: float64


In [6]:
x  = df.drop('Class',axis=1)
y=df['Class'] 

In [9]:
kf=KFold(n_splits=5,shuffle=True,random_state=42)
for fold,(train_idx,val_idx) in enumerate(kf.split(x,y)):
    y_train,y_val = y.iloc[train_idx],y.iloc[val_idx]
    print(f"\nFold{fold+1}:")
    print(f"Train:{Counter(y_train)}")
    print(f"Val: {Counter(y_val)}")


Fold1:
Train:Counter({0: 227451, 1: 394})
Val: Counter({0: 56864, 1: 98})

Fold2:
Train:Counter({0: 227446, 1: 399})
Val: Counter({0: 56869, 1: 93})

Fold3:
Train:Counter({0: 227449, 1: 397})
Val: Counter({0: 56866, 1: 95})

Fold4:
Train:Counter({0: 227455, 1: 391})
Val: Counter({0: 56860, 1: 101})

Fold5:
Train:Counter({0: 227459, 1: 387})
Val: Counter({0: 56856, 1: 105})


In [10]:
nan_indices = y[y.isna()].index
x_cleaned = x.drop(nan_indices)
y_cleaned = y.drop(nan_indices)

skf = StratifiedKFold(n_splits=5,shuffle = True,random_state=42)
for fold, (train_idx,val_idx)in enumerate(skf.split(x_cleaned,y_cleaned)):
    y_train,y_val = y_cleaned.iloc[train_idx],y_cleaned.iloc[val_idx]
    print(f"\nFold{fold+1}:")
    print(f"Train:{Counter(y_train)}")
    print(f"Val:{Counter(y_val)}")
    print(f"Val %:{y_val.value_counts(normalize=True).values*100}")


Fold1:
Train:Counter({0: 227452, 1: 393})
Val:Counter({0: 56863, 1: 99})
Val %:[99.82619992  0.17380008]

Fold2:
Train:Counter({0: 227452, 1: 393})
Val:Counter({0: 56863, 1: 99})
Val %:[99.82619992  0.17380008]

Fold3:
Train:Counter({0: 227452, 1: 394})
Val:Counter({0: 56863, 1: 98})
Val %:[99.82795246  0.17204754]

Fold4:
Train:Counter({0: 227452, 1: 394})
Val:Counter({0: 56863, 1: 98})
Val %:[99.82795246  0.17204754]

Fold5:
Train:Counter({0: 227452, 1: 394})
Val:Counter({0: 56863, 1: 98})
Val %:[99.82795246  0.17204754]


In [11]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score

scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)

In [19]:
nan_indices_array = (
    nan_indices.to_numpy()
    if hasattr(nan_indices, "to_numpy")
    else nan_indices
)
x_scaled_cleaned = np.delete(x_scaled,nan_indices_array,axis=0)
for fold, (train_idx, val_idx) in enumerate(skf.split(x_scaled_cleaned,y_cleaned)):
    print(f"\n--- Fold {fold+1}---")
    x_train,x_val = x_scaled_cleaned[train_idx],x_scaled_cleaned[val_idx]
    y_train,y_val = y_cleaned.iloc[train_idx],y_cleaned.iloc[val_idx]

    print(f"Train distribution: {np.bincount(y_train.astype(int))}")
    print(f"Val distribution: {np.bincount(y_val.astype(int))}")


--- Fold 1---
Train distribution: [227452    393]
Val distribution: [56863    99]

--- Fold 2---
Train distribution: [227452    393]
Val distribution: [56863    99]

--- Fold 3---
Train distribution: [227452    394]
Val distribution: [56863    98]

--- Fold 4---
Train distribution: [227452    394]
Val distribution: [56863    98]

--- Fold 5---
Train distribution: [227452    394]
Val distribution: [56863    98]


In [20]:
model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)

In [21]:
model.fit(x_train,y_train )

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [22]:
y_pred = model.predict(x_val)
y_prob=model.predict_proba(x_val)[:,1]

In [24]:
fold_results=[]
precision = precision_score(y_val,y_pred)
recall = recall_score(y_val,y_pred)
f1 = f1_score(y_val,y_pred)

print(f"Precision:{precision:.3f}")
print(f"Recall:{recall:.3f}")
print(f"F1-Score:{f1:.3f}")

fold_results.append({
    'fold':fold+1,
    'precision':precision,
    'recall':recall,
    'f1':f1,
})

Precision:0.065
Recall:0.867
F1-Score:0.120


In [28]:
print("Cross-Validation Summary")
results_df = pd.DataFrame(fold_results)
print(results_df)
print(f"\nMean F1-Score:{results_df['f1'].mean():.3f}(+/-{results_df['f1'].std():.3f})")
print(f"Mean Recall: {results_df['recall'].mean():.3f}(+/-{results_df['recall'].std():.3f})")
#print(f"Mean Roc-Auc: {results_df['roc_auc'].mean():.3f}(+/-{results_df['roc_auc'].std():.3f})")

Cross-Validation Summary
   fold  precision    recall        f1
0     5    0.06459  0.867347  0.120226

Mean F1-Score:0.120(+/-nan)
Mean Recall: 0.867(+/-nan)


In [33]:
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)
from sklearn.ensemble import RandomForestClassifier


def stratified_cv_pipeline(model, x, y, n_splits=5, random_state=42):
    """
    Generic stratified CV pipeline for imbalanced classification.
    """
    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    metrics = {
        'accuracy': [],
        'precision': [],
        'recall': [],
        'f1': []
    }

    for fold, (train_idx, val_idx) in enumerate(skf.split(x, y)):
        print(f"Fold {fold + 1}")

        fold_model = clone(model)

        x_train, x_val = x[train_idx], x[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        fold_model.fit(x_train, y_train)

        y_pred = fold_model.predict(x_val)

        metrics['accuracy'].append(accuracy_score(y_val, y_pred))
        metrics['precision'].append(precision_score(y_val, y_pred))
        metrics['recall'].append(recall_score(y_val, y_pred))
        metrics['f1'].append(f1_score(y_val, y_pred))

    return {k: (np.mean(v), np.std(v)) for k, v in metrics.items()}


model = RandomForestClassifier(
    class_weight='balanced',
    random_state=42
)

results = stratified_cv_pipeline(
    model,
    x_scaled_cleaned,
    y_cleaned.values
)

print(results)

Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
{'accuracy': (np.float64(0.9995330170834251), np.float64(6.80836365966564e-05)), 'precision': (np.float64(0.9572115796056746), np.float64(0.029750276776254587)), 'recall': (np.float64(0.7642547928262214), np.float64(0.02137211509565456)), 'f1': (np.float64(0.8497354089095236), np.float64(0.02168383505939144))}
